# Working with Census Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/04-census-data.ipynb)

This notebook teaches you how to retrieve and analyze US Census demographic data:

- Getting census block groups
- Retrieving demographic variables
- Three query methods
- Aggregating and analyzing data
- Practical applications

## Setup

In [ ]:
!pip install -q socialmapper[routing]

In [ ]:
import os
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

# For production, set your Census API key:
# os.environ["CENSUS_API_KEY"] = "your-key-here"

from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data
)
print("Ready!")

## Census Geography Overview

Census data is organized hierarchically:

```
Nation
  └── State
        └── County
              └── Census Tract
                    └── Block Group  ← SocialMapper uses this level
                          └── Block
```

**Block Groups** are the smallest geographic unit with detailed demographic data.

## Getting Census Block Groups

In [ ]:
# Get census blocks around a location
blocks = get_census_blocks(
    location=(47.6062, -122.3321),  # Seattle
    radius_km=2
)

print(f"Found {len(blocks)} census block groups")

# Examine one block
block = blocks[0]
print(f"\nBlock structure:")
print(f"  GEOID: {block['geoid']}")
print(f"  State FIPS: {block['state_fips']}")
print(f"  County FIPS: {block['county_fips']}")
print(f"  Tract: {block['tract']}")
print(f"  Block Group: {block['block_group']}")
print(f"  Area: {block['area_sq_km']:.2f} km²")

## Getting Blocks from an Isochrone

In [ ]:
# Create an isochrone
isochrone = create_isochrone("Portland, OR", travel_time=15)

# Get census blocks within the isochrone
blocks = get_census_blocks(polygon=isochrone)

print(f"Census block groups in 15-min drive from Portland: {len(blocks)}")

# Total area
total_area = sum(b['area_sq_km'] for b in blocks)
print(f"Total area covered: {total_area:.2f} km²")

## Retrieving Census Data

### Available Variables

| Common Name | Description |
|-------------|-------------|
| `population` | Total population |
| `median_income` | Median household income |
| `median_age` | Median age |
| `total_households` | Total households |
| `housing_units` | Total housing units |
| `median_rent` | Median gross rent |
| `median_home_value` | Median home value |

In [ ]:
# Get GEOIDs from blocks
geoids = [b['geoid'] for b in blocks]

# Retrieve census data
result = get_census_data(
    location=geoids,
    variables=["population", "median_income"]
)

print(f"Query type: {result.location_type}")
print(f"Year: {result.query_info['year']}")
print(f"Block groups: {len(result.data)}")

# Show first few results
print("\nSample data:")
for geoid, data in list(result.data.items())[:3]:
    pop = data.get('population', 'N/A')
    income = data.get('median_income', 'N/A')
    print(f"  {geoid}: pop={pop}, income=${income:,}" if isinstance(income, int) else f"  {geoid}: pop={pop}, income={income}")

## Three Ways to Query Census Data

In [ ]:
# Method 1: By polygon (isochrone)
isochrone = create_isochrone("Denver, CO", travel_time=15)
result1 = get_census_data(
    location=isochrone,
    variables=["population"]
)
print(f"Method 1 (polygon): {result1.location_type}, {len(result1.data)} blocks")

# Method 2: By GEOID list
sample_geoids = list(result1.data.keys())[:5]
result2 = get_census_data(
    location=sample_geoids,
    variables=["population"]
)
print(f"Method 2 (geoids): {result2.location_type}, {len(result2.data)} blocks")

# Method 3: By point
result3 = get_census_data(
    location=(39.7392, -104.9903),  # Denver coordinates
    variables=["population"]
)
print(f"Method 3 (point): {result3.location_type}, {len(result3.data)} blocks")

## Aggregating Population Data

In [ ]:
# Create isochrone and get census data
isochrone = create_isochrone("Chicago, IL", travel_time=20)
result = get_census_data(isochrone, variables=["population"])

# Sum population across all block groups
total_pop = sum(
    data.get("population", 0) or 0
    for data in result.data.values()
)

print(f"Total population in 20-min drive from Chicago: {total_pop:,}")
print(f"Block groups analyzed: {len(result.data)}")
print(f"Average per block group: {total_pop // len(result.data):,}")

## Calculating Averages

In [ ]:
# Get median income data
result = get_census_data(isochrone, variables=["median_income"])

# Extract valid incomes
incomes = [
    data["median_income"]
    for data in result.data.values()
    if data.get("median_income") and data["median_income"] > 0
]

if incomes:
    print(f"Income Statistics:")
    print(f"  Block groups with data: {len(incomes)}")
    print(f"  Minimum: ${min(incomes):,}")
    print(f"  Maximum: ${max(incomes):,}")
    print(f"  Average: ${sum(incomes)//len(incomes):,}")

## Population-Weighted Average

In [ ]:
# Get both population and income
result = get_census_data(
    isochrone,
    variables=["population", "median_income"]
)

# Calculate population-weighted average income
total_pop = 0
weighted_income = 0

for data in result.data.values():
    pop = data.get("population") or 0
    income = data.get("median_income")
    
    if pop and income and income > 0:
        total_pop += pop
        weighted_income += pop * income

if total_pop > 0:
    avg_income = weighted_income / total_pop
    print(f"Population-weighted average income: ${avg_income:,.0f}")
    print(f"Total population: {total_pop:,}")

## Demographic Profile Analysis

In [ ]:
def demographic_profile(location, travel_time=15):
    """Generate a demographic profile for an area."""
    
    # Create area of interest
    isochrone = create_isochrone(location, travel_time=travel_time)
    
    # Get multiple variables
    result = get_census_data(
        isochrone,
        variables=["population", "median_income", "median_age", "total_households"]
    )
    
    # Aggregate data
    stats = {
        "population": 0,
        "households": 0,
        "incomes": [],
        "ages": []
    }
    
    for data in result.data.values():
        if data.get("population"):
            stats["population"] += data["population"]
        if data.get("total_households"):
            stats["households"] += data["total_households"]
        if data.get("median_income") and data["median_income"] > 0:
            stats["incomes"].append(data["median_income"])
        if data.get("median_age") and data["median_age"] > 0:
            stats["ages"].append(data["median_age"])
    
    # Print report
    print(f"\nDemographic Profile: {location}")
    print(f"({travel_time}-minute drive)")
    print("=" * 40)
    print(f"Total Population: {stats['population']:,}")
    print(f"Total Households: {stats['households']:,}")
    print(f"Block Groups: {len(result.data)}")
    
    if stats["incomes"]:
        print(f"\nIncome Range: ${min(stats['incomes']):,} - ${max(stats['incomes']):,}")
        print(f"Average Median Income: ${sum(stats['incomes'])//len(stats['incomes']):,}")
    
    if stats["ages"]:
        print(f"\nAge Range: {min(stats['ages']):.1f} - {max(stats['ages']):.1f} years")
        print(f"Average Median Age: {sum(stats['ages'])/len(stats['ages']):.1f} years")
    
    return stats

# Generate profile
stats = demographic_profile("Austin, TX", travel_time=15)

## Comparing Locations

In [ ]:
locations = ["Seattle, WA", "Portland, OR", "San Francisco, CA"]

print("City Comparison (15-min drive):")
print("=" * 50)

for loc in locations:
    iso = create_isochrone(loc, travel_time=15)
    result = get_census_data(iso, variables=["population", "median_income"])
    
    total_pop = sum(
        d.get("population", 0) or 0
        for d in result.data.values()
    )
    
    incomes = [
        d["median_income"]
        for d in result.data.values()
        if d.get("median_income") and d["median_income"] > 0
    ]
    avg_income = sum(incomes) // len(incomes) if incomes else 0
    
    print(f"\n{loc}:")
    print(f"  Population: {total_pop:,}")
    print(f"  Avg Median Income: ${avg_income:,}")
    print(f"  Block Groups: {len(result.data)}")

## Income Inequality Analysis

In [ ]:
import statistics

def analyze_income_inequality(location):
    """Analyze income inequality in an area."""
    
    isochrone = create_isochrone(location, travel_time=20)
    result = get_census_data(isochrone, variables=["median_income", "population"])
    
    # Extract income data
    income_data = [
        {"income": d["median_income"], "pop": d["population"]}
        for d in result.data.values()
        if d.get("median_income") and d["median_income"] > 0
        and d.get("population") and d["population"] > 0
    ]
    
    if not income_data:
        print("No income data available")
        return
    
    incomes = [d["income"] for d in income_data]
    
    print(f"\nIncome Inequality Analysis: {location}")
    print("=" * 45)
    print(f"Block groups analyzed: {len(incomes)}")
    print(f"\nDistribution:")
    print(f"  Minimum: ${min(incomes):,}")
    print(f"  Maximum: ${max(incomes):,}")
    print(f"  Mean: ${statistics.mean(incomes):,.0f}")
    print(f"  Median: ${statistics.median(incomes):,.0f}")
    
    if len(incomes) > 1:
        print(f"  Std Dev: ${statistics.stdev(incomes):,.0f}")
    
    # Income ratio
    ratio = max(incomes) / min(incomes)
    print(f"\nInequality Indicators:")
    print(f"  Income ratio (max/min): {ratio:.1f}x")
    
    # Calculate percentiles
    sorted_incomes = sorted(incomes)
    n = len(sorted_incomes)
    p10 = sorted_incomes[n // 10]
    p90 = sorted_incomes[9 * n // 10]
    print(f"  90/10 ratio: {p90/p10:.1f}x")

# Analyze
analyze_income_inequality("San Francisco, CA")

## Combining Census Data with Geography

In [ ]:
# Get blocks with geography
isochrone = create_isochrone("Boston, MA", travel_time=15)
blocks = get_census_blocks(polygon=isochrone)

# Get census data
geoids = [b['geoid'] for b in blocks]
result = get_census_data(geoids, variables=["population", "median_income"])

# Combine data with geography
for block in blocks:
    data = result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0
    block['median_income'] = data.get('median_income', 0) or 0
    
    # Calculate density
    area = block['area_sq_km'] if block['area_sq_km'] > 0 else 0.01
    block['density'] = block['population'] / area

# Show top 5 by population density
print("Top 5 Block Groups by Population Density:")
print("=" * 50)

top_blocks = sorted(blocks, key=lambda x: x['density'], reverse=True)[:5]
for b in top_blocks:
    print(f"  {b['geoid']}: {b['density']:.0f} people/km²")
    print(f"    Population: {b['population']:,}, Area: {b['area_sq_km']:.2f} km²")

## Handling Missing Data

In [ ]:
# Census data sometimes has missing values
isochrone = create_isochrone("Detroit, MI", travel_time=15)
result = get_census_data(isochrone, variables=["median_income"])

# Count data quality
valid = 0
missing = 0
suppressed = 0

for geoid, data in result.data.items():
    income = data.get("median_income")
    if income is None:
        missing += 1
    elif income < 0:
        suppressed += 1  # Census suppresses data for privacy
    else:
        valid += 1

total = len(result.data)
print(f"Data Quality Report:")
print(f"  Total block groups: {total}")
print(f"  Valid data: {valid} ({valid/total*100:.1f}%)")
print(f"  Missing: {missing} ({missing/total*100:.1f}%)")
print(f"  Suppressed: {suppressed} ({suppressed/total*100:.1f}%)")

## Exercise: Demographic Comparison

In [ ]:
def compare_demographics(location1, location2, travel_time=15):
    """Compare demographics between two locations."""
    
    results = {}
    
    for loc in [location1, location2]:
        iso = create_isochrone(loc, travel_time=travel_time)
        census = get_census_data(
            iso,
            variables=["population", "median_income", "median_age"]
        )
        
        pop = sum(d.get("population", 0) or 0 for d in census.data.values())
        incomes = [d["median_income"] for d in census.data.values() 
                   if d.get("median_income") and d["median_income"] > 0]
        ages = [d["median_age"] for d in census.data.values() 
                if d.get("median_age") and d["median_age"] > 0]
        
        results[loc] = {
            "population": pop,
            "avg_income": sum(incomes) / len(incomes) if incomes else 0,
            "avg_age": sum(ages) / len(ages) if ages else 0,
            "blocks": len(census.data)
        }
    
    # Print comparison
    print(f"\nDemographic Comparison ({travel_time}-min drive)")
    print("=" * 55)
    print(f"{'Metric':<25} {location1:<15} {location2:<15}")
    print("-" * 55)
    
    r1, r2 = results[location1], results[location2]
    print(f"{'Population':<25} {r1['population']:>12,} {r2['population']:>12,}")
    print(f"{'Avg Median Income':<25} ${r1['avg_income']:>11,.0f} ${r2['avg_income']:>11,.0f}")
    print(f"{'Avg Median Age':<25} {r1['avg_age']:>12.1f} {r2['avg_age']:>12.1f}")
    print(f"{'Block Groups':<25} {r1['blocks']:>12} {r2['blocks']:>12}")
    
    return results

# Compare two cities
compare_demographics("Miami, FL", "Minneapolis, MN")

## Next Steps

Continue with:

- **[Mapping & Visualization](05-mapping-visualization.ipynb)** - Visualize census data on maps
- **[Complete Workflow](06-complete-workflow.ipynb)** - Full analysis combining all features
- **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Apply demographics to equity analysis